In [ ]:
import sqlite3
import time
import csv
from datetime import datetime
from dataclasses import dataclass

DB = 'lab3_blockchain.db'

def conn():
    c = sqlite3.connect(DB, timeout=5)
    c.execute("PRAGMA journal_mode=WAL")
    c.row_factory = sqlite3.Row
    return c

In [ ]:
with conn() as c:
    c.execute("""
        CREATE TABLE IF NOT EXISTS event_stream (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            type TEXT,
            entity_id TEXT,
            processed BOOLEAN DEFAULT 0,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

In [ ]:
class Block:
    def __init__(self, id, view):
        self.id = id
        self.view = view

    def __str__(self):
        return f"{{'id': '{self.id}', 'view': {self.view}}}"

class Vote:
    def __init__(self, block_id):
        self.block_id = block_id

    def __hash__(self):
        return hash(self.block_id)

In [ ]:
class ChainBuilder:
    def __init__(self):
        self.chain = []
        self.votes = set() 
        self.pending = {}

    def process_vote(self, vote):
        self.votes.add(vote.block_id)
        if vote.block_id in self.pending:
            self.try_add_block(self.pending[vote.block_id])

    def process_block(self, block):
        self.pending[block.id] = block
        self.try_add_block(block)

    def try_add_block(self, block):
        if block.view == len(self.chain) and block.id in self.votes:
            self.chain.append(block)
            if block.id in self.pending:
                del self.pending[block.id]
            self.check_pending()
            return True
        return False

    def check_pending(self):
        added = True
        while added:
            added = False
            for bid in list(self.pending.keys()):
                blk = self.pending[bid]
                if blk.view == len(self.chain) and blk.id in self.votes:
                    self.chain.append(blk)
                    del self.pending[bid]
                    added = True
                    break

    def run(self, inputs):
        for item in inputs:
            if isinstance(item, Vote):
                self.process_vote(item)
            else:
                self.process_block(item)
        return self.chain

In [ ]:
class BlockDB:
    @staticmethod
    def insert(id, view):
        with conn() as c:
            c.execute("INSERT OR IGNORE INTO BLOCKS (id, view, desc, img) VALUES (?,?,?,?)",
                      (id, view, '', None))
            c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)", ('block', id))

class VoteDB:
    @staticmethod
    def insert(cls, block_id, voter_id=1, source_id=1):
        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with conn() as c:
            c.execute("INSERT OR IGNORE INTO VOTES (block_id, voter_id, timestamp, source_id)VALUES (?, ?, ?, ?)", 
                      (block_id, voter_id, now, source_id))
            entity_id = f"{block_id}:{voter_id}"
            c.execute("INSERT INTO event_stream (type, entity_id) VALUES (?,?)", ('vote', entity_id))

In [ ]:
def add_block_from_csv_row(row):
    BlockDB.insert(row['block_id'], int(row['view']))

def add_vote_from_csv_row(row):
    block_id = row['block_id'].strip()
    VoteDB.insert(block_id)

def load_csv(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            typ = row['type'].strip().lower()
            if typ == 'block':
                add_block_from_csv_row(row)
            elif typ == 'vote':
                add_vote_from_csv_row(row)
    print(f"CSV {filepath} loaded")

In [ ]:
class BlockProcessor:
    def __init__(self):
        self.builder = ChainBuilder()

    def process_one_event(self, event):
        typ = event['type']
        eid = event['entity_id']
        if typ == 'block':
            with conn() as c:
                row = c.execute("SELECT id, view FROM BLOCKS WHERE id=?", (eid,)).fetchone()
                if row:
                    block = Block(row['id'], row['view'])
                    self.builder.process_block(block)
        elif typ == 'vote':
            block_id, voter_id = eid.split(':')
            with conn() as c:
                row = c.execute("SELECT block_id FROM VOTES WHERE block_id=? AND voter_id=?", (block_id, voter_id)).fetchone()
                if row:
                    vote = Vote(block_id)
                    self.builder.process_vote(vote)

    def process_all_pending(self):
        with conn() as c:
            events = c.execute("SELECT id, type, entity_id FROM event_stream WHERE processed=0").fetchall()
        for ev in events:
            self.process_one_event(ev)
            with conn() as c:
                c.execute("UPDATE event_stream SET processed=1 WHERE id=?", (ev['id'],))

    def get_chain(self):
        return self.builder.chain

In [ ]:
load_csv('lab2.csv')

proc = BlockProcessor()
proc.process_all_pending()

print("Chain:")
for b in proc.get_chain():
    print(b)

In [ ]:
proc = BlockProcessor()
while True:
    proc.process_all_pending()
    print("Очікування нових подій...")
    time.sleep(5)

In [ ]:
BlockDB.insert('0x999', 5)
VoteDB.insert('0x999')